In [ ]:
import numpy as np
import sympy as sm
import pandas as pd

import scipy as sp
from scipy.integrate import solve_ivp
import scipy.optimize as optimize

import matplotlib as mp
import matplotlib.pyplot as plt

import os
from tqdm import tqdm
import glob

# SIR model

In [ ]:
def dydt(t, y, par):

    par = np.abs(par)

    alpha = par[0]
    beta  = par[1]

    S = y[0]
    I = y[1]
    R = y[2]
    C = y[3]

    dS =  -beta*S*I
    dI = beta*S*I - alpha*I
    dR = alpha*I
    dC = beta*S*I

    return np.array([dS,dI,dR,dC])

# Solve the ODE model using solve_ivp


def SolveModel(par,tspan,init_cond,t_eval):

    ode_soln = solve_ivp(dydt,
                         tspan,
                         init_cond,
                         t_eval = t_eval,
                         args = (par,))
    return ode_soln

In [ ]:
# Define initial conditions
N0 = 1000
I0 = 5
R0 = 0
C0 = 0
S0 = N0 - I0 - R0
init_cond = np.array([S0, I0, R0, C0])

#Define time variables
t_final=100
tspan = (1,t_final)
t_eval = np.linspace(1., t_final, t_final)
tdata = np.arange(0,t_final)

### Import parameters combos to use for simulations

In [ ]:
params_df = pd.read_csv('../data/params_combos.csv')


Plot ODE solns for all combos

In [ ]:
for idx, row in params_df.iterrows():
    alpha = row["Alpha"]
    beta = row["Beta"]
    par = (alpha, beta, N0)

    soln = SolveModel(par,tspan,init_cond,t_eval)
    I = soln.y[1]
    plt.plot(tdata, I,  label='I')

## CTMC data

In [ ]:
def sim_SIR_CTMC(par):
    alpha = par[0]
    beta = par[1]

    S0, I0, R0, C0 = 995, 5, 0, 0
    x = np.array([[S0, I0, R0, C0]])
    t = [1.0]

    maxsim = 20000000
    tmax = 100
    report_time = 2
    t_report = [1]
    x_report = [x[0]]

    transition_matrix = np.array([
        [-1, 1, 0, 1],  # infection
        [0, -1, 1, 0]   # recovery
    ])

    for i in range(1, maxsim):
        S, I, R, C = x[-1]
        transition_rates = np.array([beta * S * I, alpha * I])
        sum_rates = np.sum(transition_rates)

        if sum_rates > 0:
            transition_probabilities = transition_rates / sum_rates
            interevent_time = np.random.exponential(1 / sum_rates)
            t_next = t[-1] + interevent_time
            t.append(t_next)

            u = np.random.uniform()
            cum_probs = np.cumsum(transition_probabilities)
            which_event = np.argmax(cum_probs > u)
            x_current = x[-1] + transition_matrix[which_event]
            x = np.vstack([x, x_current])

            while report_time <= t_next and report_time <= tmax:
                t_report.append(report_time)
                x_report.append(x[-1])
                report_time += 1
        else:
            while report_time <= tmax:
                t_report.append(report_time)
                x_report.append(x_report[-1])
                report_time += 1
            break

        if report_time > tmax:
            break

        if i == maxsim - 1:
            print("Warning: maxsim reached.")
            break

    return np.array(t_report), np.array(x_report)

### Simulate 100 trajectories for each parameter combination

In [ ]:
#DO NOT RUN THIS AGAIN
n_trajectories = 1000
output_dir = '../data/regenerated_ctmc_trajectories'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
#DO NOT RUN THIS AGAIN
for idx, row in tqdm(params_df.iterrows(), total=len(params_df), desc="Running simulations"):
    alpha = row["Alpha"]
    beta = row["Beta"]
    par = (alpha, beta)
    filename = f"alpha{alpha:.2f}_beta{beta:.4f}_1000".replace('.', 'p')
    all_trajectories = []

    for i in range(n_trajectories):
        t_report, x_report = sim_SIR_CTMC(par)
        df = pd.DataFrame(x_report, columns=["S", "I", "R", "C"])
        df["time"] = t_report
        df["trajectory"] = i
        all_trajectories.append(df)

    full_df = pd.concat(all_trajectories, ignore_index=True)
    full_df.to_csv(os.path.join(output_dir, f"{filename}.csv"), index=False)


# Plot trajectories

In [ ]:
params_df["R0"] = (params_df["Beta"] * 1000 / params_df["Alpha"]).round(2) #Calculate R0
params_df = params_df.sort_values("R0").reset_index(drop=True) #Sort df by R0

# Load trajectory files and map to params_df
data_dir = '../data/ctmc_trajectories' #Load data
csv_files = sorted(glob.glob(os.path.join(data_dir, '*.csv')))
param_to_file = {}
for file in csv_files:
    name = os.path.basename(file).replace('.csv', '').replace("alpha", "").replace("beta", "")
    parts = name.split("_")
    alpha = float(parts[0].replace("p", "."))
    beta = float(parts[1].replace("p", "."))
    param_to_file[(alpha, beta)] = file

In [ ]:
# Plot settings
num_plots = len(params_df)
num_cols = 4
num_rows = int(np.ceil(num_plots / num_cols))

fig, ax = plt.subplots(num_rows, num_cols, figsize=(18,12))
fig.tight_layout(pad=4.0)
ax = ax.flatten()

for idx, row in params_df.iterrows():
    alpha = row["Alpha"]
    beta = row["Beta"]
    R0 = row["R0"]
    par = (alpha, beta)

    # Load CTMC data
    file = param_to_file.get((alpha, beta))
    if file:
        df = pd.read_csv(file)

        for traj_id in df["trajectory"].unique():
            df_traj = df[df["trajectory"] == traj_id]
            ax[idx].plot(df_traj["time"], df_traj["I"])

        # ODE solution
        soln = SolveModel(par, tspan, init_cond, t_eval)
        I = soln.y[1]
        ax[idx].plot(t_eval, I, c='black', linewidth=2, label='ODE')

        ax[idx].set_title(f'$\\alpha$ = {alpha}, $\\beta$ = {beta}', fontsize=14)
        ax[idx].text(0.95, 0.85, f'$R_0$ = {R0}', fontsize=12,
                     transform=ax[idx].transAxes, ha='right')

# Hide unused subplots
for j in range(idx + 1, len(ax)):
    fig.delaxes(ax[j])

fig.suptitle("CTMC Simulations with ODE Overlay (Sorted by R₀)", fontsize=18)
fig.supxlabel("Time", fontsize=16)
fig.supylabel("Infected (I)", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Find best fit set for each parameter combination

In [ ]:
def objective_fun(par,tdata,ydatap,init_cond):

    tspan = (tdata[0],tdata[-1]);
    t_eval = tdata;

    # Solve the model with the parameters in par
    soln = SolveModel(par,tspan,init_cond,t_eval)


    # Extract the solution for new number infected over time
    I = soln.y[1]

    # Compute the SSR1
    SSR = np.sum((I-ydatap)**2)

    # Return the SSR for this parameter set and data.
    return SSR

Create folder to save these in

In [ ]:
#DO NOT RUN THIS AGAIN

best_fit_dir = '../data/regenerated_best_fit_params'
os.makedirs(best_fit_dir, exist_ok=True)

In [ ]:
#DO NOT RUN THIS AGAIN

# Optimization method and time vector
meth = 'Nelder-Mead'
tdata = np.arange(1, t_final + 1)

# Loop over param combinations again
for idx, row in params_df.iterrows():
    alpha = row["Alpha"]
    beta = row["Beta"]
    true_pars = [alpha, beta]
    par_key = (alpha, beta)

    # Load matching CTMC file
    file = param_to_file.get(par_key)
    if file:
        df = pd.read_csv(file)
        num_sims = df["trajectory"].nunique()

        # Initialize storage
        best_pars = np.zeros((2, num_sims))

        for i in range(num_sims):
            df_traj = df[df["trajectory"] == i]
            ydata = df_traj["I"].values

            result = optimize.minimize(
                objective_fun,
                true_pars,
                args=(tdata, ydata, init_cond),
                method=meth
            )

            # Store result (abs to avoid negative alpha/beta)
            best_pars[:, i] = np.abs(result.x)

        # Save to CSV
        out_name = f"best_par_alpha{alpha}_beta{beta}".replace('.', 'p')
        np.savetxt(
            f"{best_fit_dir}/{out_name}.csv",
            best_pars,
            delimiter=","
        )

# Old Plots

In [ ]:
#define subplots
fig, ax = plt.subplots(numsteps, numsteps, figsize=(20,12)) #20,14
fig.tight_layout()

#legend_labels = ['Line 1', 'Line 2', 'Line 3', '4', '5']

#create subplots
for i in np.arange(0,numsteps):
    for j in np.arange(0,numsteps):
        ax[i, j].plot(tdata,CTMC_prev[t_final*i:t_final*(i+1),num_sims*j:num_sims*(j+1)])
        ax[i,j].set_title('$\\alpha$=' + str(alpha_vec[i])+' $\\beta$= ' + str(beta_vec[j]), fontsize =14)
        params = [alpha_vec[i], beta_vec[j]]
        soln = SolveModel(params,tspan,init_cond,t_eval)
        I = soln.y[1]
        ax[i, j].plot(tdata, I, c='black', linewidth = 2, label =' ')
        ax[i, j].text(0.85, 0.85, '$R_0$ = ' + str(np.around(beta_vec[j] * N0 / alpha_vec[i], 1)),
                      fontsize=16, transform=ax[i, j].transAxes, ha='right')
#fig.text(0, 0.5, 'Y-axis Label', va='center', rotation='vertical',  fontsize=16)
#fig.text(0.5, 0, 'x-axis Label', va='center', rotation='horizontal',  fontsize=16)

In [ ]:
std_prev_CTMC=np.zeros(shape=(len(tdata)*numsteps,numsteps))
scaled_std_prev_CTMC =np.zeros(shape=(len(tdata)*numsteps,numsteps))
for i in np.arange(0,numsteps):
    for j in np.arange(0,numsteps):
        for k in np.arange(0,len(tdata)):
            std_prev_CTMC[k+t_final*i,j]=np.std(CTMC_prev[k+t_final*i,j*num_sims:(j+1)*num_sims])
            scaled_std_prev_CTMC[k+t_final*i,j] = std_prev_CTMC[k+t_final*i,j]/np.mean(CTMC_prev[k+t_final*i,j*num_sims:(j+1)*num_sims])

In [ ]:
#define subplots
fig, ax = plt.subplots(5, 5, figsize=(20,14))
fig.tight_layout()

#create subplots
for i in np.arange(0,numsteps):
    for j in np.arange(0,numsteps):
        ax[i,j].bar(range(len(tdata)), scaled_std_prev_CTMC[t_final*i:t_final*(i+1),j])
        ax[i,j].set_title('alpha=' + str(alpha_vec[i])+' beta= ' + str(beta_vec[j]), fontsize =15)
        ax[i,j].set_ylim(0, 5)
        params = [alpha_vec[i], beta_vec[j]]
        soln = SolveModel(params,tspan,init_cond,t_eval)
        I = soln.y[1]
        ax[i,j].plot(tdata, I/200, c='orange', linewidth = 3)

In [ ]:
#define subplots
fig, ax = plt.subplots(5, 5, figsize=(20,14))
fig.tight_layout()

#create subplots
for i in np.arange(0,numsteps):
    for j in np.arange(0,numsteps):
        ax[i, j].bar(range(len(tdata)), std_prev_CTMC[t_final*i:t_final*(i+1),j])
        ax[i,j].set_title('alpha=' + str(alpha_vec[i])+' beta= ' + str(beta_vec[j]), fontsize =15)
        ax[i,j].set_ylim(0, 150)
        params = [alpha_vec[i], beta_vec[j]]
        soln = SolveModel(params,tspan,init_cond,t_eval)
        I = soln.y[1]
        ax[i,j].plot(tdata, I/5, c='orange', linewidth = 3)

In [ ]:
#define subplots
fig, ax = plt.subplots(numsteps, numsteps, figsize=(20,12))
fig.tight_layout()

#create subplots
for i in np.arange(0,numsteps):
    for j in np.arange(0,numsteps):
        params = [alpha_vec[i], beta_vec[j]]
        soln = SolveModel(params,tspan,init_cond,t_eval)
        I = soln.y[1]
        ax[i, j].scatter(I, scaled_std_prev_CTMC[t_final*i:t_final*(i+1),j])
        ax[i,j].set_title('$\\alpha$=' + str(alpha_vec[i])+' $\\beta$= ' + str(beta_vec[j]), fontsize =14)
        ax[i, j].text(0.85, 0.85, '$R_0$ = ' + str(np.around(beta_vec[j] * N0 / alpha_vec[i], 1)),
                      fontsize=16, transform=ax[i, j].transAxes, ha='right')
        #ax[i, j].set_xlabel('I')
        #ax[i, j].set_ylabel('Scaled std')
        ax[i,j].set_ylim(0, 1.6)